## PART 1. Setup — Dependencies & API Clients


In [ ]:
# INSTALL IN ACTIVE NOTEBOOK KERNEL
%pip install python-dotenv google-genai openai anthropic matplotlib pyreadstat

In [28]:
# Core utilities
from dotenv import load_dotenv
import os
import time
import re
import json

# Data + modeling
import importlib
import pyreadstat
import config
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from scipy.spatial.distance import jensenshannon
from collections import defaultdict
import matplotlib.pyplot as plt

# Gemini API SDK
import google.genai as genai
from google.genai import types

# ChatGPT API SDK
from openai import OpenAI

# Claude API SDK
from anthropic import Anthropic

print("Imports ready:", pd.__version__)

# Load environment variables once.
load_dotenv()

# API keys and local endpoint config (kept in one place for easier debugging)
GEMINI_API_KEY = os.environ.get('GEMINI_API_KEY')
OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY')
ANTHROPIC_API_KEY = os.environ.get('ANTHROPIC_API_KEY')

# Llama local API config: these can be set in the .env file or default to these values for easy local testing.
LLAMA_BASE_URL = os.environ.get('LLAMA_BASE_URL', 'http://localhost:1234/v1')
LLAMA_API_KEY = os.environ.get('LLAMA_API_KEY')
LLAMA_TUNED_8B_ID = os.environ.get('LLAMA_TUNED_8B_ID', 'llama-3.1-swallow-8b-instruct-v0.2-i1')
LLAMA_BASE_8B_ID = os.environ.get('LLAMA_BASE_8B_ID', 'meta-llama-3.1-8b-instruct')

# Client registry: initialize available clients and keep missing ones as None.
_llama_client = OpenAI(base_url=LLAMA_BASE_URL, api_key=LLAMA_API_KEY)
clients = {
    'gemini':     genai.Client(api_key=GEMINI_API_KEY)     if GEMINI_API_KEY    else None,
    'chatgpt':    OpenAI(api_key=OPENAI_API_KEY)            if OPENAI_API_KEY    else None,
    'claude':     Anthropic(api_key=ANTHROPIC_API_KEY)      if ANTHROPIC_API_KEY else None,
    'llama_tuned_8b':      _llama_client,   # Llama-3.1-Swallow-8B-Instruct (Japanese-tuned)
    'llama_base_8b': _llama_client,   # Llama-3.1-8B-Instruct (untuned base) — same endpoint, different model loaded
}

print("Client status:")
for name, c in clients.items():
    print(f"- {name}: {'ready' if c is not None else 'missing API key'}")


Imports ready: 3.0.2
Client status:
- gemini: ready
- chatgpt: ready
- claude: ready
- llama_tuned_8b: ready
- llama_base_8b: ready


## PART 2. Configuration


In [30]:
# ---------------------------------------------------------------------------
# RF feature selection
# ---------------------------------------------------------------------------
K_ACCUM          = 10     # top features accumulated per RF model (fixed, per German paper)
MIN_VALID_RATE   = 0.75   # drop RF pool vars with fewer valid responses than this
TOP_K_VALUES     = [2, 4, 8, 16, 32, 64, 128]  # persona sizes to build and compare

# Variables that measure the same concept — any slice containing more than one triggers a warning
COLLINEARITY_GROUPS = {
    "Father's occupation": ["ppjxxe08", "ppjbxx15"],
    "Income (respondent)": ["szincoma", "szincomx"],
    "Income (spouse)":     ["ssszinca", "ssszincm"],
}

# ---------------------------------------------------------------------------
# Persona & simulation DEFAULTS
# ---------------------------------------------------------------------------
ACTIVE_PERSONA_K     = 2      # which top-k slice feeds the simulation (must be in TOP_K_VALUES)
MAX_PERSONAS         = 266     # safety cap on API calls — set to None for a full run (2,660)
SAMPLE_SEED          = 42


# ACTIVE_QUESTION_KEYS = ['op4trust', 'stalllf', 'q5gveqaa']  # subset of ALL_QUESTIONS to run

ACTIVE_QUESTION_KEYS = [
    'op4trust', 'tr3cgmnz', 'tr3bcraz',        # Trust
    'qfnrincr', 'q4samesm', 'op7gdevo',        # Ethnocentrism
    'q7wwhhx',  'q7jbmmcc', 'q7mgcc',          # Gender & family norms
    'q5gveqaa', 'opincdif', 'opnucpol',        # Social inequality
    'stalllf',  'nofutr',   'sfmhdprs', 'op5happz',  # Wellbeing
    'opnbmtcn', 'wllive',   'mempltgp',        # Community & civic
]


# ---------------------------------------------------------------------------
# LLM models
# ---------------------------------------------------------------------------
MODEL_CONFIG = {
    'gemini':     {'model_id': 'gemini-2.5-flash',  'sleep_seconds': 1.0, 'temperature': 0.7, 'max_tokens': 100},
    'chatgpt':    {'model_id': 'gpt-4o-mini',       'sleep_seconds': 1.0, 'temperature': 0.7, 'max_tokens': 100},
    'claude':     {'model_id': 'claude-haiku-4-5',  'sleep_seconds': 1.0, 'temperature': 0.7, 'max_tokens': 100},
    'llama_tuned_8b':      {'model_id': LLAMA_TUNED_8B_ID,      'sleep_seconds': 0.0, 'temperature': 0.7, 'max_tokens': 100},  # Llama-3.1-Swallow-8B (Japanese-tuned)
    'llama_base_8b': {'model_id': LLAMA_BASE_8B_ID, 'sleep_seconds': 0.0, 'temperature': 0.7, 'max_tokens': 100},  # Llama-3.1-8B-Instruct (untuned base)
}

# Approximate USD cost per 1M tokens — update if provider pricing changes
COST_PER_1M = {
    'gemini':     {'input': 0.075, 'output': 0.30},
    'chatgpt':    {'input': 0.150, 'output': 0.60},
    'claude':     {'input': 0.800, 'output': 4.00},
    'llama_tuned_8b':      {'input': 0.000, 'output': 0.00},
    'llama_base_8b': {'input': 0.000, 'output': 0.00},
}

# ---------------------------------------------------------------------------
# System prompt
# ---------------------------------------------------------------------------
# Minimal role-setter. The actual task framing (perspective-taking, question,
# answer options, completion cue) is built into the user prompt — see Part 8.
sys_instruct = (
    "あなたは日本の社会調査（JGSS）の回答者です。"
    "以下のペルソナの視点に立って、質問に対する回答を選択肢から選んでください。"
    "回答は選択肢の番号（数字）のみを出力してください。説明や理由は不要です。"
)

print("Configuration loaded.")
print(f"  Active persona: TOP-{ACTIVE_PERSONA_K}  |  cap: {MAX_PERSONAS or 'none (full run)'}  |  questions: {ACTIVE_QUESTION_KEYS}")
print(f"  Models: {list(MODEL_CONFIG.keys())}")

Configuration loaded.
  Active persona: TOP-2  |  cap: 266  |  questions: ['op4trust', 'tr3cgmnz', 'tr3bcraz', 'qfnrincr', 'q4samesm', 'op7gdevo', 'q7wwhhx', 'q7jbmmcc', 'q7mgcc', 'q5gveqaa', 'opincdif', 'opnucpol', 'stalllf', 'nofutr', 'sfmhdprs', 'op5happz', 'opnbmtcn', 'wllive', 'mempltgp']
  Models: ['gemini', 'chatgpt', 'claude', 'llama_tuned_8b', 'llama_base_8b']


## PART 3. Data


In [ ]:
importlib.reload(config)

DATA_PATH = 'data/20172018_data_jap.sav'

df, meta = pyreadstat.read_sav(DATA_PATH)
df.columns = [c.strip() for c in df.columns]

# Replace JGSS soft missing codes with NaN (9, 99, 999 are "no answer" conventions)
for code in config.MISSING_CODES:
    df.replace(code, np.nan, inplace=True)

# Variable + value label lookups — used by Part 6 to build human-readable personas instead of raw numeric codes
var_labels   = dict(zip(meta.column_names, meta.column_labels))   # var code → description
value_labels = meta.variable_value_labels                          # var code → {numeric value → text}

config.sanity_check(df.columns)
print(f"\nLoaded: {len(df)} respondents × {len(df.columns)} variables")
print(f"Label coverage: {sum(1 for v in var_labels.values() if v)} column labels, "
      f"{len(value_labels)} variables with value labels")


Variable classification check passed:
  Total in dataset:    558
  Excluded:            30
  Core demographics:   10
  Outcomes:            19 across 6 categories
  RF pool (the rest):  499

Loaded: 2660 respondents × 558 variables
Label coverage: 558 column labels, 552 variables with value labels


## PART 4. Load Checkpoint
Restore RF ranking, personas, and prior model results/metrics from disk.
Run this once at the start of every session — then skip straight to Part 5.
If `data/personas_by_k.json` or `data/rf_ranking_filtered.csv` does not yet exist, run Appendix B + C once to build them.

In [6]:
# ---------------------------------------------------------------------------
# LOAD — restore artefacts from previous sessions without re-running them.
# Loads results for ACTIVE_PERSONA_K specifically; metrics_all.csv loads
# the full history across all k-values.
# ---------------------------------------------------------------------------
CKPT_DIR = 'results'

# 1. RF ranking
_path = f'{CKPT_DIR}/rf_ranking_filtered.csv'
if os.path.exists(_path):
    rf_ranking_df = pd.read_csv(_path)
    sorted_ranking_filtered = list(zip(rf_ranking_df['variable'], rf_ranking_df['importance_score']))
    print(f"Loaded RF ranking:   {_path}  ({len(sorted_ranking_filtered)} variables)")
else:
    print(f"Not found (skipped): {_path}")

# 2. Personas by k
_path = f'{CKPT_DIR}/personas_by_k.json'
if os.path.exists(_path):
    with open(_path, encoding='utf-8') as f:
        _raw = json.load(f)
    personas_by_k = {int(k): v for k, v in _raw.items()}
    personas = personas_by_k[ACTIVE_PERSONA_K]
    print(f"Loaded personas:     {_path}  (k={list(personas_by_k.keys())}, active=TOP-{ACTIVE_PERSONA_K})")
else:
    print(f"Not found (skipped): {_path}")

# 3. Per-model results for the active k
results_by_model = {}
for model_label in MODEL_CONFIG:
    _path = f'{CKPT_DIR}/results_{model_label}_k{ACTIVE_PERSONA_K}.csv'
    if os.path.exists(_path):
        results_by_model[model_label] = pd.read_csv(_path, index_col=0)
        print(f"Loaded results:      {_path}  {results_by_model[model_label].shape}")
    else:
        print(f"Not found (skipped): {_path}")

# 4. Full metrics history across all batches
_path = f'{CKPT_DIR}/metrics_all.csv'
if os.path.exists(_path):
    metrics_df = pd.read_csv(_path)
    metrics_rows = metrics_df.to_dict('records')
    runs = metrics_df.groupby(['model', 'persona_k']).size().reset_index(name='questions')
    print(f"Loaded metrics:      {_path}  ({len(metrics_df)} rows across {len(runs)} completed runs)")
    print(runs.to_string(index=False))
else:
    metrics_df = pd.DataFrame()
    metrics_rows = []
    print(f"Not found (skipped): {_path}  — metrics_df is empty")

# 5. Show run log if available
_path = f'{CKPT_DIR}/run_log.csv'
if os.path.exists(_path):
    print(f"\nRun log ({_path}):")
    print(pd.read_csv(_path).to_string(index=False))
else:
    print(f"\nNo run log yet.")


Loaded RF ranking:   results/rf_ranking_filtered.csv  (162 variables)
Loaded personas:     results/personas_by_k.json  (k=[2, 4, 8, 16, 32, 64, 128], active=TOP-2)
Not found (skipped): results/results_gemini_k2.csv
Not found (skipped): results/results_chatgpt_k2.csv
Not found (skipped): results/results_claude_k2.csv
Not found (skipped): results/results_llama_k2.csv
Not found (skipped): results/metrics_all.csv  — metrics_df is empty

No run log yet.


## PART 5. Simulation Build


In [33]:
# ---------------------------------------------------------------------------
# Full JGSS-2017/2018 question bank (all 19 outcome variables).
# Completion-style layout: question text and answer options are kept SEPARATE,
# ACTIVE_QUESTION_KEYS (set in Configuration) selects which questions actually run.
# ---------------------------------------------------------------------------
# Question text — human-readable wording shown to the LLM.
# These are paraphrases; verify against the JGSS codebook for publication fidelity.
QUESTION_TEXT = {
    # --- Trust ---
    'op4trust': "一般的に言って、人は信頼できると思いますか？",
    'tr3cgmnz': "国会議員をどの程度信頼していますか？",
    'tr3bcraz': "省庁・政府機関をどの程度信頼していますか？",
    # --- Ethnocentrism ---
    'qfnrincr': "日本の外国人を増やすべきだと思いますか？",
    'q4samesm': "同性婚についてどうお考えですか？",
    'op7gdevo': "人間の本質について、1（非常に自己中心的）から7（非常に善良）のどこに当てはまりますか？",
    # --- Gender & family norms ---
    'q7wwhhx':  "「男性は外で働き、女性は家庭を守るべきだ」という考えに同意しますか？",
    'q7jbmmcc': "「母親が働くと子どもに悪影響がある」という考えに同意しますか？",
    'q7mgcc':   "「結婚したカップルは子どもを持つべきだ」という考えに同意しますか？",
    # --- Social inequality ---
    'q5gveqaa': "政府は所得格差を縮小する責任があると思いますか？",
    'opincdif': "所得格差は大きくなりすぎていると思いますか？",
    'opnucpol': "原子力政策についてどうお考えですか？",
    # --- Wellbeing ---
    'stalllf':  "現在の生活全体にどの程度満足していますか？",
    'nofutr':   "将来に希望が持てないと感じますか？",
    'sfmhdprs': "過去4週間で、気分が落ち込んだり憂うつになったりすることがありましたか？",
    'op5happz': "あなたは幸福だと思いますか？",
    # --- Community & civic ---
    'opnbmtcn': "近所の人々は互いに気にかけ合っていると思いますか？",
    'wllive':   "現在住んでいる地域に住み続けたいと思いますか？",
    'mempltgp': "政治団体や市民運動に参加していますか？",
}

# Build ALL_QUESTIONS dynamically from the .sav's value_labels.
# This guarantees the options shown to the LLM are byte-identical to what
# real respondents saw — essential for a valid JSD comparison.
_MISSING_CODES_SET = set(config.MISSING_CODES)

def _strip_duplicate_code_prefix(code, label):
    """JGSS labels like '1：人間の本性は本来「悪」である' for code 1, or bare '2'
    for an unlabeled intermediate point, would render as '1: 1：…' or '2: 2'
    when prefixed with their code. Strip the duplicate prefix."""
    label = str(label).strip()
    cleaned = re.sub(rf'^{int(code)}[:：]\s*', '', label)
    return cleaned

ALL_QUESTIONS = {}
for _qvar, _q_text in QUESTION_TEXT.items():
    _raw = value_labels.get(_qvar, {})
    _valid = sorted([(int(k), _strip_duplicate_code_prefix(k, v))
                     for k, v in _raw.items() if int(k) not in _MISSING_CODES_SET])
    # Bare-number labels (e.g. '2' on a 7-point scale) become empty after stripping;
    # render those as just the digit instead of "2: ".
    _options = [f"{c}: {lab}" if lab else str(c) for c, lab in _valid]
    ALL_QUESTIONS[_qvar] = {
        'question':  _q_text,
        'options':   _options,
        'scale_min': _valid[0][0],
        'scale_max': _valid[-1][0],
    }

question_bank = {key: ALL_QUESTIONS[key] for key in ACTIVE_QUESTION_KEYS}

# Save the full question bank to JSON so visualization.ipynb can load scale info
with open('results/question_bank.json', 'w', encoding='utf-8') as _f:
    json.dump(ALL_QUESTIONS, _f, ensure_ascii=False, indent=2)

# Precompute real distributions once and reuse across models
real_distributions = {
    var_name: df[var_name]
        .dropna()
        .astype(int)
        .value_counts(normalize=True)
        .sort_index()
        .reindex(range(spec['scale_min'], spec['scale_max'] + 1), fill_value=0)
    for var_name, spec in question_bank.items()
}

# One DataFrame per model
# Conditional init — preserves state loaded from Part 5 (Load Checkpoint)
if 'results_by_model' not in globals(): results_by_model = {}
if 'metrics_rows'     not in globals(): metrics_rows = []
if 'metrics_df'       not in globals(): metrics_df = pd.DataFrame()

# Token usage accumulator — reset per model run, updated by each generate function
if 'token_usage' not in globals():
    token_usage = {m: {'input': 0, 'output': 0, 'calls': 0} for m in MODEL_CONFIG}


def build_user_prompt(persona_json, spec):
    """Builds the per-respondent prompt. Persona attributes first, then the
    question and a clean newline-separated list of options. The model is
    instructed (via sys_instruct) to reply with just the digit."""
    options_str = "\n".join(spec['options'])
    return (
        f"以下の特徴を持つ人物の視点に立ってください：{persona_json}\n\n"
        f"次の質問に対するこの人物の回答はどの選択肢ですか：{spec['question']}\n\n"
        f"回答選択肢：\n{options_str}"
    )


def extract_first_int(raw_text, scale_min, scale_max):
    """Extract the answer digit from raw_text.
    Returns the LAST in-scale digit found — reasoning preambles often cite
    out-of-scale numbers (ages, codes) before the final answer, so 'last
    in-range' is more robust than 'first overall'."""
    if not raw_text:
        return None
    in_range = [int(m) for m in re.findall(r"\d+", raw_text)
                if scale_min <= int(m) <= scale_max]
    return in_range[-1] if in_range else None


def reset_model_outputs(model_label):
    """Clear stored results, metrics, and token counts for this model."""
    global metrics_rows
    results_by_model.pop(model_label, None)
    metrics_rows = [r for r in metrics_rows if r.get('model') != model_label]
    token_usage[model_label] = {'input': 0, 'output': 0, 'calls': 0}


def record_metrics(model_label, var_name, simulated_answers, all_values, persona_k):
    real_dist = real_distributions[var_name]
    sim_series = pd.Series(simulated_answers).dropna().astype(int)
    sim_dist = sim_series.value_counts(normalize=True).reindex(all_values, fill_value=0)
    valid_n = int(sim_series.notna().sum())

    if sim_dist.sum() == 0:
        jsd_score = None
        print(f"  {var_name}: no valid responses — JSD skipped")
    else:
        jsd_score = float(jensenshannon(real_dist, sim_dist))
        print(f"  {var_name}: JSD={jsd_score:.4f}  valid={valid_n}/{len(simulated_answers)}")

    return {
        'model': model_label,
        'question_var': var_name,
        'persona_k': persona_k,
        'valid_answers': valid_n,
        'total_personas': len(simulated_answers),
        'jsd': jsd_score,
    }


def print_token_summary(model_label):
    u = token_usage[model_label]
    if u['calls'] == 0:
        return
    rates = COST_PER_1M.get(model_label, {'input': 0, 'output': 0})
    cost = u['input'] / 1e6 * rates['input'] + u['output'] / 1e6 * rates['output']
    avg_in = u['input'] / u['calls']
    print(f"\nToken usage — {model_label}:")
    print(f"  Calls:      {u['calls']:,}")
    print(f"  Input:      {u['input']:,} tokens  (avg {avg_in:.0f} / call)")
    print(f"  Output:     {u['output']:,} tokens")
    print(f"  Est. cost:  ${cost:.4f}  "
          f"(@ ${rates['input']}/1M in, ${rates['output']}/1M out)")


def save_model_results(model_label):
    """Persist this model's results, metrics, and run-log entry to disk.
    Called automatically at the end of run_model_simulation."""
    import datetime
    CKPT_DIR = 'results'

    if model_label not in results_by_model:
        print(f"  No results for {model_label} — nothing to save.")
        return

    # 1. Per-model results CSV — keyed by (model, k) so batches never overwrite each other
    rdf = results_by_model[model_label]
    results_path = f'{CKPT_DIR}/results_{model_label}_k{ACTIVE_PERSONA_K}.csv'
    rdf.to_csv(results_path)
    print(f"  Saved: {results_path}  {rdf.shape}")

    # 2. Metrics — append/dedupe by (model, k) so re-runs replace prior rows
    new_metrics = metrics_df[metrics_df['model'] == model_label]
    metrics_path = f'{CKPT_DIR}/metrics_all.csv'
    if os.path.exists(metrics_path):
        existing = pd.read_csv(metrics_path)
        mask = ~((existing['model'] == model_label) & (existing['persona_k'] == ACTIVE_PERSONA_K))
        combined = pd.concat([existing[mask], new_metrics], ignore_index=True)
    else:
        combined = new_metrics.copy()
    combined.to_csv(metrics_path, index=False)
    print(f"  Saved: {metrics_path}  ({len(new_metrics)} new rows, {len(combined)} total)")

    # 3. Run log — one row per (model, k) batch
    log_path = f'{CKPT_DIR}/run_log.csv'
    log_row = {
        'timestamp':   datetime.datetime.now().strftime('%Y-%m-%d %H:%M'),
        'model':       model_label,
        'persona_k':   ACTIVE_PERSONA_K,
        'questions':   ','.join(ACTIVE_QUESTION_KEYS),
        'n_personas':  MAX_PERSONAS or len(personas),
        'sample_seed': SAMPLE_SEED,
        'mean_jsd':    round(new_metrics['jsd'].mean(), 4),
        'valid_rate':  round((new_metrics['valid_answers'] / new_metrics['total_personas']).mean(), 3),
    }
    new_log = pd.DataFrame([log_row])
    if os.path.exists(log_path):
        pd.concat([pd.read_csv(log_path), new_log], ignore_index=True).to_csv(log_path, index=False)
    else:
        new_log.to_csv(log_path, index=False)
    print(f"  Saved: {log_path}")


def run_model_simulation(model_label, generate_fn):
    global metrics_rows, metrics_df
    reset_model_outputs(model_label)

    # Random-sample respondents (reproducible across models & k via SAMPLE_SEED)
    if MAX_PERSONAS is None or MAX_PERSONAS >= len(personas):
        sample_idx = list(range(len(personas)))
    else:
        _rng = np.random.default_rng(SAMPLE_SEED)
        sample_idx = sorted(_rng.choice(len(personas), MAX_PERSONAS, replace=False).tolist())
    active_personas = [personas[i] for i in sample_idx]
    model_results = pd.DataFrame(index=[df.index[i] for i in sample_idx])
    print(f"Sampling {len(active_personas)} respondents (seed={SAMPLE_SEED}, indices {sample_idx[:3]}…{sample_idx[-3:]})")

    for var_name, spec in question_bank.items():
        all_values = range(spec['scale_min'], spec['scale_max'] + 1)
        simulated_answers = []
        print(f"\n[{model_label}] {var_name} (scale {spec['scale_min']}–{spec['scale_max']})")

        for i, persona_json in enumerate(active_personas):
            try:
                raw = generate_fn(build_user_prompt(persona_json, spec))
                answer = extract_first_int(raw, spec['scale_min'], spec['scale_max'])
                simulated_answers.append(answer)
            except Exception as e:
                print(f"  respondent {i+1} error: {e}")
                simulated_answers.append(None)
            time.sleep(MODEL_CONFIG[model_label]['sleep_seconds'])

            if (i + 1) % 500 == 0 or (i + 1) == len(active_personas):
                done = sum(1 for a in simulated_answers if a is not None)
                print(f"  {i+1}/{len(active_personas)} personas  ({done} valid so far)")

        model_results[var_name] = simulated_answers
        metrics_rows.append(
            record_metrics(model_label, var_name, simulated_answers, all_values, ACTIVE_PERSONA_K)
        )

    results_by_model[model_label] = model_results
    metrics_df = pd.DataFrame(metrics_rows)
    print(f"\n{model_label} done.")
    print_token_summary(model_label)

    print(f"\nAuto-saving {model_label} results to disk...")
    save_model_results(model_label)


_cap = f"capped at {MAX_PERSONAS}" if MAX_PERSONAS is not None else "full run"
print(f"Active questions: {list(question_bank.keys())}")
print(f"Personas to simulate: {len(personas)}  (TOP-{ACTIVE_PERSONA_K}, {_cap})")
print(f"API calls per model:  {(MAX_PERSONAS or len(personas)) * len(question_bank)}")

Active questions: ['op4trust', 'tr3cgmnz', 'tr3bcraz', 'qfnrincr', 'q4samesm', 'op7gdevo', 'q7wwhhx', 'q7jbmmcc', 'q7mgcc', 'q5gveqaa', 'opincdif', 'opnucpol', 'stalllf', 'nofutr', 'sfmhdprs', 'op5happz', 'opnbmtcn', 'wllive', 'mempltgp']
Personas to simulate: 2660  (TOP-2, capped at 266)
API calls per model:  5054


## PART 6. LLMs


### Per-run override
Edit the values below and re-run this single cell to switch `k` or persona cap mid-session — no need to scroll back to Part 2 or re-run Part 4. Part 2 holds the session defaults; this cell overrides them for the next LLM call.

In [34]:
# === PER-RUN OVERRIDE ===  (defaults live in Part 2)
ACTIVE_PERSONA_K = 2
MAX_PERSONAS     = 266
SAMPLE_SEED = 42

personas = personas_by_k[ACTIVE_PERSONA_K]
print(f"Override active: TOP-{ACTIVE_PERSONA_K}, {MAX_PERSONAS or len(personas)} personas")

Override active: TOP-2, 266 personas


## Gemini

In [ ]:
def gemini_generate_raw_text(user_prompt):
    gemini_client = clients['gemini']
    if gemini_client is None:
        raise ValueError("GEMINI_API_KEY is missing")

    response = gemini_client.models.generate_content(
        model=MODEL_CONFIG['gemini']['model_id'],
        contents=user_prompt,
        config=types.GenerateContentConfig(
            system_instruction=sys_instruct,
            temperature=MODEL_CONFIG['gemini']['temperature'],
            max_output_tokens=MODEL_CONFIG['gemini']['max_tokens'],
        ),
    )
    u = response.usage_metadata
    token_usage['gemini']['input']  += u.prompt_token_count or 0
    token_usage['gemini']['output'] += u.candidates_token_count or 0
    token_usage['gemini']['calls']  += 1
    return response.text or ""


run_model_simulation('gemini', gemini_generate_raw_text)

## ChatGPT

In [27]:
def chatgpt_generate_raw_text(user_prompt):
    client = clients['chatgpt']
    if client is None:
        raise ValueError("OPENAI_API_KEY is missing")

    resp = client.chat.completions.create(
        model=MODEL_CONFIG['chatgpt']['model_id'],
        messages=[
            {'role': 'system', 'content': sys_instruct},
            {'role': 'user',   'content': user_prompt},
        ],
        temperature=MODEL_CONFIG['chatgpt']['temperature'],
        max_tokens=MODEL_CONFIG['chatgpt']['max_tokens'],
    )
    token_usage['chatgpt']['input']  += resp.usage.prompt_tokens
    token_usage['chatgpt']['output'] += resp.usage.completion_tokens
    token_usage['chatgpt']['calls']  += 1
    return resp.choices[0].message.content or ""


run_model_simulation('chatgpt', chatgpt_generate_raw_text)

Sampling 266 respondents (seed=42, indices [18, 51, 58]…[2631, 2640, 2646])

[chatgpt] op4trust (scale 1–4)
  266/266 personas  (266 valid so far)
  op4trust: JSD=0.3349  valid=266/266

[chatgpt] tr3cgmnz (scale 1–4)
  266/266 personas  (266 valid so far)
  tr3cgmnz: JSD=0.2350  valid=266/266

[chatgpt] tr3bcraz (scale 1–4)
  266/266 personas  (266 valid so far)
  tr3bcraz: JSD=0.1891  valid=266/266

[chatgpt] qfnrincr (scale 1–2)
  266/266 personas  (266 valid so far)
  qfnrincr: JSD=0.4649  valid=266/266

[chatgpt] q4samesm (scale 1–4)
  266/266 personas  (266 valid so far)
  q4samesm: JSD=0.2828  valid=266/266

[chatgpt] op7gdevo (scale 1–7)
  266/266 personas  (266 valid so far)
  op7gdevo: JSD=0.3834  valid=266/266

[chatgpt] q7wwhhx (scale 1–7)
  266/266 personas  (266 valid so far)
  q7wwhhx: JSD=0.2868  valid=266/266

[chatgpt] q7jbmmcc (scale 1–7)
  266/266 personas  (266 valid so far)
  q7jbmmcc: JSD=0.2686  valid=266/266

[chatgpt] q7mgcc (scale 1–7)
  266/266 personas  (266

## Claude

In [ ]:
def claude_generate_raw_text(user_prompt):
    claude_client = clients['claude']
    if claude_client is None:
        raise ValueError("ANTHROPIC_API_KEY is missing")

    response = claude_client.messages.create(
        model=MODEL_CONFIG['claude']['model_id'],
        system=sys_instruct,
        max_tokens=MODEL_CONFIG['claude']['max_tokens'],
        temperature=MODEL_CONFIG['claude']['temperature'],
        messages=[{"role": "user", "content": user_prompt}],
    )
    token_usage['claude']['input']  += response.usage.input_tokens
    token_usage['claude']['output'] += response.usage.output_tokens
    token_usage['claude']['calls']  += 1
    return "".join(block.text for block in response.content if hasattr(block, "text"))


run_model_simulation('claude', claude_generate_raw_text)

## Llama Swallow 3.1 8B (llama_tuned_8b)


In [20]:
def llama_tuned_8b_generate_raw_text(user_prompt):
    client = clients['llama_tuned_8b']
    if client is None:
        raise ValueError("Llama client is not initialized")

    resp = client.chat.completions.create(
        model=MODEL_CONFIG['llama_tuned_8b']['model_id'],
        messages=[
            {'role': 'system', 'content': sys_instruct},
            {'role': 'user',   'content': user_prompt},
        ],
        temperature=MODEL_CONFIG['llama_tuned_8b']['temperature'],
        max_tokens=MODEL_CONFIG['llama_tuned_8b']['max_tokens'],
    )
    token_usage['llama_tuned_8b']['input']  += resp.usage.prompt_tokens
    token_usage['llama_tuned_8b']['output'] += resp.usage.completion_tokens
    token_usage['llama_tuned_8b']['calls']  += 1
    return resp.choices[0].message.content or ""


run_model_simulation('llama_tuned_8b', llama_tuned_8b_generate_raw_text)

Sampling 266 respondents (seed=42, indices [18, 51, 58]…[2631, 2640, 2646])

[llama] op4trust (scale 1–4)
  266/266 personas  (266 valid so far)
  op4trust: JSD=0.0722  valid=266/266

[llama] tr3cgmnz (scale 1–4)
  266/266 personas  (266 valid so far)
  tr3cgmnz: JSD=0.1622  valid=266/266

[llama] tr3bcraz (scale 1–4)
  266/266 personas  (266 valid so far)
  tr3bcraz: JSD=0.1666  valid=266/266

[llama] qfnrincr (scale 1–2)
  266/266 personas  (266 valid so far)
  qfnrincr: JSD=0.1611  valid=266/266

[llama] q4samesm (scale 1–4)
  266/266 personas  (266 valid so far)
  q4samesm: JSD=0.1948  valid=266/266

[llama] op7gdevo (scale 1–7)
  266/266 personas  (266 valid so far)
  op7gdevo: JSD=0.2883  valid=266/266

[llama] q7wwhhx (scale 1–7)
  266/266 personas  (266 valid so far)
  q7wwhhx: JSD=0.3429  valid=266/266

[llama] q7jbmmcc (scale 1–7)
  266/266 personas  (266 valid so far)
  q7jbmmcc: JSD=0.3141  valid=266/266

[llama] q7mgcc (scale 1–7)
  266/266 personas  (266 valid so far)
  q

## Llama 3.1 8B Instruct (llama_base_8b — untuned)
Same architecture, size, and quantization (Q4_K_S) as the Swallow run above — but without the Japanese continued pretraining. Used to isolate the contribution of Japanese fine-tuning. **Load this model in LM Studio before running the cell.**

In [ ]:
def llama_base_8b_generate_raw_text(user_prompt):
    client = clients['llama_base_8b']
    if client is None:
        raise ValueError("Llama base client is not initialized")

    resp = client.chat.completions.create(
        model=MODEL_CONFIG['llama_base_8b']['model_id'],
        messages=[
            {'role': 'system', 'content': sys_instruct},
            {'role': 'user',   'content': user_prompt},
        ],
        temperature=MODEL_CONFIG['llama_base_8b']['temperature'],
        max_tokens=MODEL_CONFIG['llama_base_8b']['max_tokens'],
    )
    token_usage['llama_base_8b']['input']  += resp.usage.prompt_tokens
    token_usage['llama_base_8b']['output'] += resp.usage.completion_tokens
    token_usage['llama_base_8b']['calls']  += 1
    return resp.choices[0].message.content or ""


run_model_simulation('llama_base_8b', llama_base_8b_generate_raw_text)

Sampling 266 respondents (seed=42, indices [18, 51, 58]…[2631, 2640, 2646])

[llama_base_8b] op4trust (scale 1–4)
  266/266 personas  (266 valid so far)
  op4trust: JSD=0.2852  valid=266/266

[llama_base_8b] tr3cgmnz (scale 1–4)
  266/266 personas  (266 valid so far)
  tr3cgmnz: JSD=0.1353  valid=266/266

[llama_base_8b] tr3bcraz (scale 1–4)
  266/266 personas  (266 valid so far)
  tr3bcraz: JSD=0.2696  valid=266/266

[llama_base_8b] qfnrincr (scale 1–2)
  266/266 personas  (266 valid so far)
  qfnrincr: JSD=0.0770  valid=266/266

[llama_base_8b] q4samesm (scale 1–4)
  266/266 personas  (266 valid so far)
  q4samesm: JSD=0.1512  valid=266/266

[llama_base_8b] op7gdevo (scale 1–7)


## Appendix A. Connectivity Check


In [32]:
# Smoke test — one small prompt per client to verify connectivity.
def smoke_test_client(name, client):
    if client is None:
        return False, 'missing API key'
    try:
        if name == 'gemini':
            resp = client.models.generate_content(
                model='gemini-2.5-flash',
                contents='Reply with exactly: pong',
            )
            text = resp.text or ""
        elif name == 'chatgpt':
            resp = client.chat.completions.create(
                model='gpt-4o-mini',
                messages=[{'role': 'user', 'content': 'Reply with exactly: pong'}],
                max_tokens=10,
            )
            text = resp.choices[0].message.content or ""
        elif name == 'claude':
            resp = client.messages.create(
                model='claude-haiku-4-5',
                max_tokens=16,
                messages=[{'role': 'user', 'content': 'Reply with exactly: pong'}],
            )
            text = "".join(b.text for b in resp.content if hasattr(b, "text"))
        elif name in ('llama_tuned_8b', 'llama_base_8b'):
            model_id = MODEL_CONFIG[name]['model_id']
            resp = client.chat.completions.create(
                model=model_id,
                messages=[{'role': 'user', 'content': 'Reply with exactly: pong'}],
                max_tokens=10,
            )
            text = resp.choices[0].message.content or ""
        else:
            return False, f'no smoke test configured for {name}'
        return True, text.strip()
    except Exception as e:
        return False, f'{type(e).__name__}: {e}'


print('Client smoke test:')
for name, client in clients.items():
    ok, detail = smoke_test_client(name, client)
    print(f'  {name}: {"OK" if ok else "FAIL"} | {detail}')

Client smoke test:
  gemini: OK | pong
  chatgpt: OK | pong
  claude: FAIL | BadRequestError: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'Your credit balance is too low to access the Anthropic API. Please go to Plans & Billing to upgrade or purchase credits.'}, 'request_id': 'req_011CbGjbam9XBPaQX5TykA6H'}
  llama_tuned_8b: OK | pong
  llama_base_8b: OK | Pong


## Appendix B. RF Feature Selection
Only run once, to select the top-k relevant predictors. Should not be re-run. predictors are saved in results

In [ ]:
"""
# ---------------------------------------------------------------------------
# RF pool prep — eligible variables and red-flag list (used only by RF run).
# ---------------------------------------------------------------------------
# RF pool = all variables except EXCLUDES, CORE_DEMOGRAPHICS, and OUTCOMES (never seen by RF)
rf_pool = config.get_rf_pool(df.columns)
n = len(df)
print(f"RF pool: {len(rf_pool)} variables")

# Flag pool variables below the valid-rate threshold
red_flag = {
    var: df[var].notna().sum() / n
    for var in rf_pool
    if df[var].notna().sum() / n < MIN_VALID_RATE
}
print(f"Red-flagged (<{MIN_VALID_RATE:.0%} valid): {len(red_flag)} variables")

# Filtered RF ranking — source of all persona variable slices.
# Only variables with valid rate >= MIN_VALID_RATE enter the pool.
rf_pool_filtered = [v for v in rf_pool if df[v].notna().sum() / n >= MIN_VALID_RATE]
print(f"Filtered RF pool: {len(rf_pool_filtered)} variables  ({len(rf_pool) - len(rf_pool_filtered)} dropped below {MIN_VALID_RATE:.0%})")

imputer_f = SimpleImputer(strategy='most_frequent')
df_rf_filtered = pd.DataFrame(
    imputer_f.fit_transform(df[rf_pool_filtered]),
    columns=rf_pool_filtered,
    index=df.index,
)

master_scores_filtered = defaultdict(float)
for i, target_var in enumerate(rf_pool_filtered):
    X = df_rf_filtered.drop(columns=[target_var])
    y = df_rf_filtered[target_var].astype(str)
    rf = RandomForestClassifier(n_estimators=50, max_features='sqrt', random_state=42, n_jobs=-1)
    rf.fit(X, y)
    importances = pd.Series(rf.feature_importances_, index=X.columns)
    for feat, score in importances.nlargest(K_ACCUM).items():
        master_scores_filtered[feat] += score
    if (i + 1) % 50 == 0:
        print(f"  {i + 1}/{len(rf_pool_filtered)} done")

sorted_ranking_filtered = sorted(master_scores_filtered.items(), key=lambda x: x[1], reverse=True)

# One ranking, sliced at each top-k cutpoint
persona_slices = {
    k: [var for var, _ in sorted_ranking_filtered[:k]]
    for k in TOP_K_VALUES
}

# Display ranked list up to the largest k, with cutpoint markers
cutpoints = set(TOP_K_VALUES)
print(f"\n--- FILTERED RF RANKING (≥{MIN_VALID_RATE:.0%} valid, K_ACCUM={K_ACCUM}) ---")
for i, (var, score) in enumerate(sorted_ranking_filtered[:max(TOP_K_VALUES)]):
    rank = i + 1
    label = var_labels.get(var, "")
    rate = df[var].notna().sum() / n
    marker = f"  ◄ TOP-{rank}" if rank in cutpoints else ""
    print(f"  {rank:3d}. {var} — {label}  ({rate:.1%}){marker}")

# Collinearity check at each slice
print()
warned = False
for k in TOP_K_VALUES:
    for group_name, group_vars in COLLINEARITY_GROUPS.items():
        hits = [v for v in group_vars if v in persona_slices[k]]
        if len(hits) > 1:
            print(f"  ⚠ TOP-{k} collinearity — '{group_name}': {hits}")
            warned = True
if not warned:
    print("No collinearity conflicts across any slice.")

# ---------------------------------------------------------------------------
# SAVE — persist RF ranking so we can skip this step in future sessions.
# ---------------------------------------------------------------------------
rf_ranking_df = pd.DataFrame(sorted_ranking_filtered, columns=['variable', 'importance_score'])
rf_ranking_df['label']      = rf_ranking_df['variable'].map(var_labels)
rf_ranking_df['valid_rate'] = rf_ranking_df['variable'].map(lambda v: df[v].notna().sum() / len(df))
rf_ranking_df.to_csv('results/rf_ranking_filtered.csv', index=False)
print(f"\nSaved: data/rf_ranking_filtered.csv  ({len(rf_ranking_df)} variables)")
"""

## Appendix C. Persona Creation


In [47]:
# Build persona profiles for every top-k slice.
# Personas use human-readable labels (variable descriptions as keys,
# "(code) label" strings as values) so the LLM can actually interpret them.
# Mirrors the German paper's Appendix B.1 format: '(3) Mittlere Reife'.
# ACTIVE_PERSONA_K (set in Configuration) selects which set feeds the simulation cells.

def encode_value(var, val, value_labels):
    """Return the human-readable label for a coded value, or the raw number
    for continuous variables. JGSS numeric codes (like '(645)' for an
    occupation) are dropped — they're arbitrary identifiers, not semantic
    content, and they introduce out-of-scale digits that can confuse parsing."""
    label = value_labels.get(var, {}).get(float(val))
    if label is not None:
        return label
    return int(val) if val == int(val) else val

persona_slices = {
    k: [var for var, _ in sorted_ranking_filtered[:k]]
    for k in TOP_K_VALUES
}

personas_by_k = {}
for k in TOP_K_VALUES:
    persona_vars = config.CORE_DEMOGRAPHICS + persona_slices[k]
    persona_list = []
    for _, row in df.iterrows():
        persona_dict = {}
        for var in persona_vars:
            val = row.get(var)
            if pd.notna(val):
                key = var_labels.get(var) or var
                persona_dict[key] = encode_value(var, val, value_labels)
        persona_list.append(json.dumps(persona_dict, ensure_ascii=False))
    personas_by_k[k] = persona_list

personas = personas_by_k[ACTIVE_PERSONA_K]

print(f"Built persona sets for top-k: {TOP_K_VALUES}")
print(f"Active: TOP-{ACTIVE_PERSONA_K}  ({len(config.CORE_DEMOGRAPHICS)} core + {ACTIVE_PERSONA_K} RF vars per persona)")
print(f"\nExample persona (TOP-{ACTIVE_PERSONA_K}):")
print(personas[0])

# ---------------------------------------------------------------------------
# SAVE — persist personas so we can skip this step in future sessions.
# ---------------------------------------------------------------------------
with open('results/personas_by_k.json', 'w', encoding='utf-8') as f:
    json.dump({str(k): v for k, v in personas_by_k.items()}, f, ensure_ascii=False, indent=2)
print(f"\nSaved: data/personas_by_k.json  (k={list(personas_by_k.keys())})")


Built persona sets for top-k: [2, 4, 8, 16, 32, 64, 128]
Active: TOP-2  (10 core + 2 RF vars per persona)

Example persona (TOP-2):
{"性別": "女", "年齢": 76, "市郡規模": "人口20万人未満の市", "本人年収：全体": "70万円未満", "子どもの人数": 3, "先週の就労経験": "仕事をしていない", "初職の職種": "味噌・醤油・缶詰食品・乳製品製造工、飲食料品製造作業者", "15歳の頃の父：職種（ISCO08）": "Construction supervisors"}

Saved: data/personas_by_k.json  (k=[2, 4, 8, 16, 32, 64, 128])


## Archive — Reference Code
Code blocks below are kept for reference only and are **not part of the active pipeline**. They are wrapped in triple-quoted strings so they never execute accidentally.

In [ ]:
""" # Leave-one-out RF ranking over the full RF pool (unfiltered).
# Kept for reference — use the filtered ranking (6b90c8be) for actual persona construction.
# This version ran on the unfiltered pool (includes low-valid-rate variables) and
# produced master_importance_scores / sorted_ranking.
master_importance_scores = defaultdict(float)

for i, target_var in enumerate(rf_pool):
    X = df_rf.drop(columns=[target_var])
    y = df_rf[target_var].astype(str)
    rf = RandomForestClassifier(n_estimators=50, max_features='sqrt', random_state=42, n_jobs=-1)
    rf.fit(X, y)
    importances = pd.Series(rf.feature_importances_, index=X.columns)
    for feat, score in importances.nlargest(K_ACCUM).items():
        master_importance_scores[feat] += score
    if (i + 1) % 50 == 0:
        print(f"  {i + 1}/{len(rf_pool)} done")

sorted_ranking = sorted(master_importance_scores.items(), key=lambda x: x[1], reverse=True)

print(f"\n--- TOP 10 PERSONA VARIABLES — unfiltered RF pool (reference only) ---")
flagged_in_top_k = []
for i, (var, score) in enumerate(sorted_ranking[:10]):
    label = var_labels.get(var, "")
    flag = " *** RED FLAG" if var in red_flag else ""
    rate = red_flag.get(var, df[var].notna().sum() / len(df))
    print(f"  {i+1:2d}. {var} — {label}  (score={score:.4f}, valid={rate:.1%}){flag}")
    if var in red_flag:
        flagged_in_top_k.append(var)

if flagged_in_top_k:
    print(f"\n  {len(flagged_in_top_k)} red-flagged variable(s) in top-10.")
else:
    print("\n  No red-flagged variables in top-10.")
"""